# Initiation to the DeThermIn workflow

Use the DeThermIn workflow from a Jupyter Notebook!

Let's begin by importing the main module.

In [ ]:
import DeThermIn

You can consult the parameters by calling the following script: `DeThermIn.set_params()`. You can otherwise check the dictionary containing the parameters (but you won't have access to the description of the parameters):

In [ ]:
DeThermIn.parameters

You should especially look for the parameters `start_date`, `end_date` and `resolution` to orient the workflow towards the date range and final resolution you prefer.

If it is the first use of the workflow, you should also:
- predownload the geostationary data -> `DeThermIn.predownload()` (also if you changed the time range)
- download the geo-file containing the areas available for the workflow -> `DeThermIn.download_available_areas()`

Let's first pre-process the data, which mostly means that we have to download it. One mandatory parameter is a `pandas.GeoDataFrame` corresponding to th area of interest.

In [ ]:
from lidar_hd_tools import geodataframe_from_coordinates
gdf = geodataframe_from_coordinates(lat=52.089,lon=5.110, size=1000) # Utrecht!

Here is an example of an area of interest vs the `bbox_size` parameter later used in the pre-processing.

![](_figures/workflow_element_2.png)

In [ ]:
ancillary_dc, narrow_swath_dc, geostationary_dc = DeThermIn.pre_process(gdf)

Now that you launched the pre-processing, you might want to know what is happening.

Simultaneously, data from ECOSTRESS, Sentinel-2 and airborne LiDAR data are downloaded from different APIs. Finally, the geostationary data which has been already pre-downloaded for all the available areas, is clipped to match the greater area of interest.

![](_figures/workflow_element_1.png)
![](_figures/workflow_element_3.png)

The next step allows to fit a Diurnal Temperature Cycle to the geostationary LST.

In [ ]:
dtc_fitted_dc = DeThermIn.DTC_fitting(geostationary_dc)

This being done, it is now possible to downscale the parameters of this model to the resolution of the ECOSTRESS LST, thanks to a spatio-temporal unmixing method. This requires to choose a clear-sky day (will be prompted).

![](_figures/workflow_element_4.png)

In [ ]:
fusioned_dc = DeThermIn.spatio_temporal_fusion(narrow_swath_dc, dtc_fitted_dc)

The next and last step is the super-resolution, done thanks to a deep-learning regression. It uses extra-features computed according to the Sun's position at each available timestamp.

![](_figures/workflow_element_6.png)
![](_figures/workflow_element_5.png)

Several models have been trained, differing by the used features for training. Please consult the dedicated folder to learn more. The most sophisticated model can be used with `mode="dynamic"`.

In [ ]:
DeThermIn_dc = DeThermIn.super_resolution(ancillary_dc, fusioned_dc, mode="dynamic")

In [ ]:
DeThermIn_dc.plot("SR_LST_dynamic")

You now have the end-product of this workflow! Congrats. If you encountered issues, don't hesitate to report them.